<a href="https://colab.research.google.com/github/sparshmishra777-beep/Predictive-Maintenance-of-Defense-equipments-/blob/main/Predictive_Maintenance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas scikit-learn joblib ipywidgetsc

In [ ]:
import pandas as pd

# Load data
df = pd.read_csv("Predictive Maintenance System data set (1).csv", encoding='latin1')

# Show first rows
print("Original Data:")
display(df.head())

# Clean data
df = df.drop_duplicates()
df.fillna(method='ffill', inplace=True)

# Fix column names
df.columns = df.columns.str.strip()

print("Cleaned Data:")
display(df.head())

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Feature Engineering
df['Risk Score'] = (
    df['Temperature (°C)'] * 0.3 +
    df['Vibration (mm/s)'] * 0.4 +
    df['Component Wear Index'] * 0.3
)

# Encode Machinery Type
encoder = LabelEncoder()
df['Machinery Type'] = encoder.fit_transform(df['Machinery Type'])

# Drop unnecessary column if exists
if 'Maintenance History' in df.columns:
    df = df.drop(columns=['Maintenance History'])

display(df.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import joblib

# Split features & target
X = df.drop(columns=['Maintenance Urgency Score'])
y = df['Maintenance Urgency Score']

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

# Train model
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)

print("✅ Model Trained Successfully!")

# Save files
joblib.dump(model, "model.pkl")
joblib.dump(encoder, "encoder.pkl")
joblib.dump(scaler, "scaler.pkl")

In [ ]:
from sklearn.metrics import r2_score

y_pred = model.predict(X_test)
print("Model Accuracy (R2 Score):", r2_score(y_test, y_pred))

In [ ]:
def get_recommendation(temp, pressure, vibration, humidity, wear):
    rec = []

    # Temperature
    if temp > 80:
        rec.append("⚠️ Reduce temperature below 70°C (Cooling required)")
    elif temp < 40:
        rec.append("⚠️ Temperature too low, check system efficiency")

    # Pressure
    if pressure > 250:
        rec.append("⚠️ Reduce pressure (Risk of damage)")
    elif pressure < 80:
        rec.append("⚠️ Pressure too low, system inefficiency")

    # Vibration
    if vibration > 15:
        rec.append("⚠️ High vibration → Check alignment & lubrication")

    # Humidity
    if humidity > 70:
        rec.append("⚠️ High humidity → Risk of corrosion")

    # Wear
    if wear > 0.7:
        rec.append("⚠️ Replace worn components immediately")

    if len(rec) == 0:
        rec.append("✅ All parameters within safe range")

    return rec

In [ ]:
def predict_degradation(score):
    if score >= 80:
        return "🔴 Critical: Failure expected in 0–2 days"
    elif score >= 60:
        return "🟠 High Risk: Maintenance needed in 3–7 days"
    elif score >= 40:
        return "🟡 Moderate: Check within 1–2 weeks"
    else:
        return "🟢 Healthy: No immediate risk"

In [ ]:
ideal_values = {
    "Temperature": "50–70°C",
    "Pressure": "100–200 psi",
    "Vibration": "< 10 mm/s",
    "Humidity": "30–50%",
    "Wear Index": "< 0.5"
}

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

print("🔧 Predictive Maintenance System (Final UI)")

# -------------------------------
# Recommendation Function
# -------------------------------
def get_recommendation(temp, pressure, vibration, humidity, wear):
    rec = []

    if temp > 80:
        rec.append("⚠️ Reduce temperature below 70°C")
    if pressure > 250:
        rec.append("⚠️ Reduce pressure levels")
    if vibration > 15:
        rec.append("⚠️ Check alignment & lubrication")
    if humidity > 70:
        rec.append("⚠️ High humidity → corrosion risk")
    if wear > 0.7:
        rec.append("⚠️ Replace worn components")

    if len(rec) == 0:
        rec.append("✅ System is operating in ideal conditions")

    return rec


# -------------------------------
# Degradation Function
# -------------------------------
def predict_degradation(score):
    if score >= 80:
        return "🔴 Critical: Failure in 0–2 days"
    elif score >= 60:
        return "🟠 High Risk: Maintenance in 3–7 days"
    elif score >= 40:
        return "🟡 Moderate: Check in 1–2 weeks"
    else:
        return "🟢 Healthy Condition"


# -------------------------------
# Ideal Values
# -------------------------------
ideal_values = {
    "Temperature": "50–70°C",
    "Pressure": "100–200 psi",
    "Vibration": "< 10 mm/s",
    "Humidity": "30–50%",
    "Wear Index": "< 0.5"
}


# -------------------------------
# UI Widgets
# -------------------------------
machinery = widgets.Dropdown(
    options=list(encoder.classes_),
    description='Machine:'
)

temperature = widgets.IntSlider(min=0, max=200, value=50, description='Temp')
pressure = widgets.IntSlider(min=0, max=500, value=100, description='Pressure')
vibration = widgets.FloatSlider(min=0, max=50, value=5, description='Vibration')
humidity = widgets.IntSlider(min=0, max=100, value=40, description='Humidity')
wear = widgets.FloatSlider(min=0, max=1, value=0.2, description='Wear')

button = widgets.Button(description="Predict 🚨")
output = widgets.Output()


# -------------------------------
# Prediction Function
# -------------------------------
def predict(b):
    with output:
        output.clear_output()

        try:
            # Encode
            m = encoder.transform([machinery.value])[0]

            # Feature engineering
            risk_score = (
                temperature.value * 0.3 +
                vibration.value * 0.4 +
                wear.value * 0.3
            )

            # Input dataframe
            input_data = pd.DataFrame([[
                m,
                temperature.value,
                pressure.value,
                vibration.value,
                humidity.value,
                wear.value,
                risk_score
            ]], columns=[
                'Machinery Type',
                'Temperature (°C)',
                'Pressure (psi)',
                'Vibration (mm/s)',
                'Humidity (%)',
                'Component Wear Index',
                'Risk Score'
            ])

            # Scale
            input_scaled = scaler.transform(input_data)

            # Predict
            pred = model.predict(input_scaled)[0]

            # Extra outputs
            degradation = predict_degradation(pred)
            recs = get_recommendation(
                temperature.value,
                pressure.value,
                vibration.value,
                humidity.value,
                wear.value
            )

            # ---------------- OUTPUT ----------------
            print(f"\n🚨 Maintenance Score: {round(pred,2)}")
            print(degradation)

            print("\n💡 Recommendations:")
            for r in recs:
                print("-", r)

            print("\n🎯 Ideal Values:")
            for k, v in ideal_values.items():
                print(f"{k}: {v}")

        except Exception as e:
            print("❌ Error:", e)


# -------------------------------
# Button Click
# -------------------------------
button.on_click(predict)


# -------------------------------
# Display UI
# -------------------------------
display(
    machinery,
    temperature,
    pressure,
    vibration,
    humidity,
    wear,
    button,
    output
)